In [ ]:
"""
routing_engine.py

Ingests generated_swaths.json (output of the Geometry Engine) and computes the
shortest continuous path that visits every swath exactly once, entering each
swath from either end (whichever is cheaper), using Google OR-Tools.

Modeling approach
------------------
Each swath has physical length and two possible traversal directions:
    - direction 0: enter at `start`, exit at `end`
    - direction 1: enter at `end`,   exit at `start`

We turn every swath into TWO directional nodes in the routing graph. A
disjunction constraint forces the solver to choose exactly ONE of the two
directional nodes per swath (i.e. paint it in exactly one direction).

The "cost" of any arc in the graph is the Euclidean (in-air) distance from
the EXIT point of the node you're leaving to the ENTRY point of the node
you're arriving at. Distance travelled *inside* a swath (entry -> exit) is
never counted, since that's mandatory painting distance, not travel time.

A virtual depot node (index 0) anchors the start and end of the route.
"""

import json
import math
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp


def load_swaths(path):
    with open(path, "r") as f:
        return json.load(f)


def euclidean(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])


def build_directional_nodes(swaths):
    """
    Returns a list of directional nodes, two per swath:
        nodes[2*i]     -> swath i, direction 0 (start -> end)
        nodes[2*i + 1] -> swath i, direction 1 (end -> start)

    Each node is a dict: {'entry': (x, y), 'exit': (x, y),
                           'swath_idx': i, 'direction': 0 or 1}
    """
    nodes = []
    for i, s in enumerate(swaths):
        start = tuple(s["start"])
        end = tuple(s["end"])
        nodes.append({"entry": start, "exit": end, "swath_idx": i, "direction": 0})
        nodes.append({"entry": end, "exit": start, "swath_idx": i, "direction": 1})
    return nodes


def build_distance_matrix(nodes, depot_point, scale=1000):
    """
    Builds an (N+1) x (N+1) integer distance matrix, where index 0 is the
    depot and indices 1..N correspond to `nodes[0..N-1]`.

    OR-Tools requires integer costs, so we scale float distances up and
    round (scale=1000 preserves 3 decimal places of precision).
    """
    size = len(nodes) + 1
    matrix = [[0] * size for _ in range(size)]

    for i in range(size):
        for j in range(size):
            if i == j:
                continue

            if i == 0:
                p_from = depot_point
                p_to = nodes[j - 1]["entry"]
            elif j == 0:
                p_from = nodes[i - 1]["exit"]
                p_to = depot_point
            else:
                p_from = nodes[i - 1]["exit"]
                p_to = nodes[j - 1]["entry"]

            matrix[i][j] = int(round(euclidean(p_from, p_to) * scale))

    return matrix


def solve(swaths, depot_point, time_limit_seconds=10, verbose=True):
    nodes = build_directional_nodes(swaths)
    n_swaths = len(swaths)
    distance_matrix = build_distance_matrix(nodes, depot_point)

    manager = pywrapcp.RoutingIndexManager(len(distance_matrix), 1, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return distance_matrix[from_node][to_node]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # --- Enforce: exactly one direction of each swath is visited ---
    # AddDisjunction([a, b], penalty, max_cardinality=1) lets the solver pick
    # at most one of {a, b} for "free"; not picking either costs `penalty`.
    # A very large penalty makes skipping a swath entirely never worth it,
    # which effectively makes visiting exactly one direction mandatory.
    penalty = 10 ** 9
    for i in range(n_swaths):
        node_a = 2 * i + 1  # +1 offset: index 0 is the depot
        node_b = 2 * i + 2
        index_a = manager.NodeToIndex(node_a)
        index_b = manager.NodeToIndex(node_b)
        routing.AddDisjunction([index_a, index_b], penalty, 1)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    search_parameters.time_limit.FromSeconds(time_limit_seconds)

    solution = routing.SolveWithParameters(search_parameters)

    if solution is None:
        raise RuntimeError("No solution found. Try increasing time_limit_seconds.")

    # --- Extract the ordered path ---
    path_coords = []
    visited_swaths = []
    total_air_distance = 0.0

    index = routing.Start(0)
    prev_exit_point = depot_point

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        if node != 0:
            n = nodes[node - 1]
            total_air_distance += euclidean(prev_exit_point, n["entry"])
            path_coords.append(list(n["entry"]))
            path_coords.append(list(n["exit"]))
            visited_swaths.append({"swath_idx": n["swath_idx"], "direction": n["direction"]})
            prev_exit_point = n["exit"]
        index = solution.Value(routing.NextVar(index))

    if verbose:
        print(f"Swaths in input:   {n_swaths}")
        print(f"Swaths routed:     {len(visited_swaths)}")
        if len(visited_swaths) != n_swaths:
            missing = n_swaths - len(visited_swaths)
            print(f"WARNING: {missing} swath(s) were not routed. "
                  f"Consider raising the disjunction penalty or time_limit_seconds.")
        print(f"Total in-air (travel) distance: {total_air_distance:.3f}")

    return path_coords, visited_swaths, total_air_distance


if __name__ == "__main__":
    swaths = load_swaths("generated_swaths.json")

    # Depot: where the brush starts/ends. Defaults to the first swath's start
    # point; change this if your tool has a fixed home/parking position.
    depot = tuple(swaths[0]["start"])

    path, order, total_distance = solve(swaths, depot_point=depot)

    print("\nOrdered coordinate path:")
    for p in path:
        print(p)

    with open("optimized_path.json", "w") as f:
        json.dump(
            {
                "depot": list(depot),
                "path": path,
                "swath_order": order,
                "total_air_distance": total_distance,
            },
            f,
            indent=2,
        )
    print("\nSaved result to optimized_path.json")